In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import openpyxl
import pyodbc
import numpy as np
import os
from config import server, database, username, password
from datetime import datetime
import locale
import time
from pathlib import Path
import mysql.connector

conn = mysql.connector.connect(
    host=os.environ["MYSQL_HOST"],
    user=os.environ["MYSQL_USER"],
    password=os.environ["MYSQL_PASSWORD"],
    database=os.environ["MYSQL_DATABASE"]
)
query = "SELECT * FROM proyeccion_detalle_pedidos where id_cadena <>'123456'"
df_Oc_cliente = pd.read_sql(query, conn)
conn.close()

connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)
conn = pyodbc.connect(connection_string)

with open('my_query2.sql', 'r') as file:
    query = file.read()
df_roles = pd.read_sql(query, conn)



df_calculo = pd.read_excel('Ajustes_pedido.xlsx', sheet_name='base', skiprows=7, engine='openpyxl')
df_calculo['Sap'] = df_calculo['Sap'].astype(str)
df_calculo['Sku'] = df_calculo['Sku'].astype(str)
df_Oc_cliente['sap'] = df_Oc_cliente['sap'].astype(str)
df_Oc_cliente['sku'] = df_Oc_cliente['sku'].astype(str)

df_merged = pd.merge(
    df_calculo,
    df_Oc_cliente[['sap', 'sku', 'Piezas_pedido']],
    left_on=['Sap', 'Sku'],
    right_on=['sap', 'sku'],
    how='left'
)
df_merged.rename(columns={'Piezas_pedido': 'Pedido_original_cadena'}, inplace=True)
df_merged.drop(columns=['sap', 'sku'], inplace=True)
df_calculo = df_merged
df_calculo['Pedido_original_cadena'] = df_calculo['Pedido_original_cadena'].fillna(0)

locale.setlocale(locale.LC_TIME, 'en_US.UTF-8')
day_map = {
    'Monday': 'Lunes',
    'Tuesday': 'Martes',
    'Wednesday': 'Miercoles',
    'Thursday': 'Jueves',
    'Friday': 'Viernes',
    'Saturday': 'Sabado',
    'Sunday': 'Domingo'
}
dia_actual = day_map[datetime.today().strftime('%A')]
print(dia_actual)

df_roles['sap'] = df_roles['sap'].astype(str)
df_calculo['Sap'] = df_calculo['Sap'].astype(str)
tiendas_que_piden_hoy = df_roles[df_roles[dia_actual] == 1]['sap']
df_calculo = df_calculo[df_calculo['Sap'].isin(tiendas_que_piden_hoy)]
df_calculo = df_calculo[df_calculo['Sap'].isin(tiendas_que_piden_hoy)]

df_calculo.to_csv('resumen_pedido.csv', index=False)

def actualizar_resumen_carga(
    filename: str = 'Resumen_carga.xlsx',
    visible: bool = True,
    bring_to_front: bool = True,
    timeout_sec: int = 1800,
    update_links: int = 3
) -> bool:
    try:
        import win32com.client as win32
    except ImportError as e:
        raise ImportError("Falta pywin32. Instálalo con: pip install pywin32") from e

    wb_path = (Path.cwd() / filename).resolve()
    if not wb_path.exists():
        raise FileNotFoundError(f"No encontré el archivo: {wb_path}")

    xl = win32.DispatchEx("Excel.Application")
    xl.Visible = bool(visible)
    xl.DisplayAlerts = False
    try:
        try:
            xl.ScreenUpdating = True
        except Exception:
            pass

        wb = xl.Workbooks.Open(str(wb_path), UpdateLinks=update_links, ReadOnly=False)

        if visible and bring_to_front:
            try:
                xl.WindowState = -4143
                try:
                    xl.ActiveWindow.Activate()
                except Exception:
                    pass
            except Exception:
                pass

        try:
            for conn in wb.Connections:
                for attr in ("OLEDBConnection", "ODBCConnection"):
                    try:
                        obj = getattr(conn, attr)
                        obj.BackgroundQuery = False
                    except Exception:
                        pass
        except Exception:
            pass

        for ws in wb.Worksheets:
            try:
                for qt in ws.QueryTables:
                    try:
                        qt.BackgroundQuery = False
                    except Exception:
                        pass
            except Exception:
                pass
            try:
                for lo in ws.ListObjects:
                    try:
                        qt = lo.QueryTable
                        qt.BackgroundQuery = False
                    except Exception:
                        pass
            except Exception:
                pass

        wb.RefreshAll()
        try:
            xl.CalculateUntilAsyncQueriesDone()
        except Exception:
            pass

        start = time.time()
        while True:
            refreshing = False
            try:
                for conn in wb.Connections:
                    try:
                        if hasattr(conn, 'OLEDBConnection') and conn.OLEDBConnection.Refreshing:
                            refreshing = True
                            break
                        if hasattr(conn, 'ODBCConnection') and conn.ODBCConnection.Refreshing:
                            refreshing = True
                            break
                    except Exception:
                        pass
            except Exception:
                pass

            try:
                if int(xl.CalculationState) != 0:
                    refreshing = True
            except Exception:
                pass

            if not refreshing:
                break
            if time.time() - start > timeout_sec:
                raise TimeoutError("Se agotó el tiempo esperando que Excel termine de actualizar.")
            time.sleep(1.5)

        wb.Save()
        #wb.Close(SaveChanges=True)
        print("Comparación finalizada ✅")
        return True

    finally:
        #try:
            #xl.Quit()
        #except Exception:
                pass

try:
    ok = actualizar_resumen_carga(
        filename='Resumen_carga.xlsx',
        visible=True,
        bring_to_front=True,
        timeout_sec=1800,
        update_links=3
    )
    print("Estado:", ok)
except Exception as e:
    import traceback
    print("❌ Falló actualizar_resumen_carga:", e)
    traceback.print_exc()

In [2]:
df_Oc_cliente.head(5)

,fecha,id_cadena,Piezas_pedido,sap,sku,user
0,2026-03-20 06:27:09,667537,18,312175,211,desconocido
1,2026-03-20 06:27:09,667537,18,312175,236,desconocido
2,2026-03-20 06:27:09,667537,24,312175,954,desconocido
3,2026-03-20 06:27:09,667537,12,312175,7434,desconocido
4,2026-03-20 06:27:09,667537,24,312175,8206,desconocido
